<a href="https://colab.research.google.com/github/martinabergonzoni/ANALISI-DEI-PILOTI-DI-FORMULA-1---STAGIONE-2008/blob/main/Analisi_dei_Piloti_del_Mondiale_di_Formula_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Obiettivo del progetto**

Analizzare i risultati del Campionato Mondiale di Formula 1 della stagione 2008



------



**Struttura del dataset**

**Nome**: *formula1_data.csv*

**Informazioni di colonna**:
1. ***Driver***: Nome del pilota
2. ***Team*** - Nome del costruttore per il quale il pilota gareggia
3. ***Race***: Città in cui si è svolto il Gran Premio
4. ***Country***: Paese in cui si è svolto il Gran Premio
5. ***Position***: Numero compreso tra 0 e 8 che indica l'ordine di arrivo del pilota   (0 significa che il pilota non è arrivato tra i primi 8 e non ha ottenuto punti)



---



**Sistema di punteggio**

Al termine di ogni Gran Premio, i punti vengono assegnati ai piloti in base al loro ordine di arrivo come segue:
- 1° posto: 10 punti
- 2° posto: 8 punti
- 3° posto: 6 punti
- 4° posto: 5 punti
- 5° posto: 4 punti
- 6° posto: 3 punti
- 7° posto: 2 punti
- 8° posto: 1 punto
- 9° posto o oltre: 0 punti



---



# Funzione per l'analisi delle performance individuali dei piloti

La prima funzione riceve in input il nome di un pilota e restituisce una lista contenente tre informazioni chiave:
- Il totale dei punti accumulati dal pilota durante il campionato.
- Il numero di vittorie, ovvero quante volte il pilota è arrivato primo in un Gran Premio.
- Il numero di podi, ovvero quante volte il pilota è arrivato tra i primi tre classificati.

Questa funzione sarà utile per analizzare le performance individuali dei piloti e avere una chiara visione delle loro posizioni nel corso della stagione.

In [ ]:
#importo i moduli che mi servono
import csv
import os

#carico il dataset all'interno dell'ambiente
from google.colab import files
uploaded = files.upload()
nome_file = list(uploaded.keys())[0]

#apro il file
def apri_file(nome_file):
  with open(nome_file, "r", encoding="utf-8") as csv_file:
      csv_reader = csv.DictReader(csv_file)
      return list(csv_reader)

#creo il dizionario contenete il sistema di punteggio
punteggio = {
    1 : 10,
    2 : 8,
    3 : 6,
    4 : 5,
    5 : 4,
    6 : 3,
    7 : 2,
    8 : 1,
    9 : 0
}

#creo una funzione che mi consenta di associare i punti alla posizione
def punti_da_posizione(posizione):
    try:
        posizione = int(posizione)  # converte la stringa in intero
        return punteggio.get(posizione, 0)  # restituisce i punti o 0 se non presente
    except ValueError:
        return 0  # se la posizione non si può convertire perchè, ad esempio, è vuota, restituisce 0

dati = apri_file(nome_file) #con la variabile dati posso utilizzare il file

nome_pilota = input("Inserisci il nome del pilota: ").strip().title() #chiedo all'utente il nome del pilota che vuole analizzare e normalizzo l'output

def performance(nome_pilota: str): #definisco la funzione performance che prende in input il nome del pilot da analizzare
  tot_punti = 0 #inizializzo a 0 la variabile che conta i punti fatti
  tot_vittorie = 0 #inizializzo a 0 la variabile che conta le vittorie (1* posto)
  tot_podi = 0 #inizializzo a 0 la variabile che conta i podi raggiunti (tra 1* e 3* posto)


  for row in dati: #leggo ogni riga del file
    if row["Driver"].strip().title() == nome_pilota: #se nella riga i-esima trovo il nome del pilota che cerco, mi fermo
      posizione = int(row["Position"]) #trovo nella riga la posizione a cui è arrivato
      tot_punti += punti_da_posizione(posizione) #aggiungo al contatore dei punti il punteggio relativo
      if posizione == 1:
        tot_vittorie += 1 #se è arrivato 1°, incremento di 1 il contatore delle vittorie
      if posizione in [1, 2, 3]:
        tot_podi += 1 #se è arrivato tra il 1° e 3° posto, incremento di 1 il contatore dei podi
  print(f"Punti: {tot_punti}")
  print(f"Podi: {tot_podi}")
  print(f"Vittorie: {tot_vittorie}")

performance(nome_pilota)


Saving formula1_data (2).csv to formula1_data (2).csv
Inserisci il nome del pilota: Hamilton
Punti: 98
Podi: 10
Vittorie: 5


#Funzione per la creazione della classifica finale dei piloti
La seconda funzione genera un dizionario contenente i nomi dei piloti come chiavi e il loro punteggio totale come valori. Il dizionario viene poi utilizzato per creare una classifica generale dei piloti.

Infine, la classifica sarà salvata in un file di testo (Drivers_Standings_2008.txt) con il seguente formato:

*Drivers Standings 2008 Formula 1*

*NomePilota1: PunteggioTotale*

*NomePilota2: PunteggioTotale*

In [ ]:
#definisco la funzione per creare la classifica dei piloti
def classifica_piloti(dati):
  punteggi = {} #creo un dizionario vuoto

  for row in dati:
    pilota = row.get("Driver", "").strip().title() #estraggo il nome del pilota
    try:
      posizione = int(row.get("Position", "")) #estraggo la relativa posizione di arrivo e la converto ad intero per usarla nella funzione punti_da_posizione
    except ValueError: #nel caso in cui il valore sia, ad esempio, vuoto, passo alla riga successiva
      continue
    punti = punti_da_posizione(posizione) #ricavo il punteggio ottenuto grazie alla posizione raggiunta
    punteggi[pilota] = punteggi.get(pilota, 0) + punti #associo alla chiave pilota il punteggio relativo e lo itero per ogni riga

  classifica = sorted(punteggi.items(), key=lambda x: x[1], reverse=True) #ordino il dizionario punteggi in senso descrescente per ottenere una classifica

  return classifica


#definisco la funzione per salvare la classifica in un file di testo
def salva_classifica_piloti(classifica, nome_file):
    with open(nome_file, "w", encoding="utf-8") as file: #creo il file in modalità scrittura
        if not classifica: #gestisce il caso in cui la classifica risulta vuota
          file.write("Nessun dato disponibile.\n") #comunica che non ci sono dati disponibili ed esce
          return
        file.write("Drivers Standings 2008 Formula 1\n\n") #inserisco l'intestazione
        for pilota, punti in classifica:
            file.write(f"{pilota}: {punti} \n") #prendo piloti e punti dalla classifica e formatto la scrittura sulla base della richiesta del compito

classifica = classifica_piloti(dati)
salva_classifica_piloti(classifica, "Classifica_drivers_2008.txt")

from google.colab import files
files.download("Classifica_drivers_2008.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Funzione per la classifica dei costruttori
La terza funzione crea un dizionario con i nomi dei team/costruttori come chiavi e il loro punteggio totale come valori. Il punteggio di ciascun team è la somma dei punti ottenuti dai piloti che hanno gareggiato per quel costruttore.

Questa funzione utilizza i dati precedentemente generati per i piloti e calcola la classifica dei costruttori. Anche questa informazione è essenziale per avere una visione chiara delle prestazioni dei team durante l'anno.

In [ ]:
def classifica_team(dati):
    punteggi_team = {}

    for row in dati:
        team = row.get("Team", "").strip().title()  #estrae il nome del team e lo normalizza
        try:
            posizione = int(row.get("Position", "")) #se la posizione esiste, la estrae
        except ValueError:
            continue  #se la posizione ha dei problemi, la salta

        punti = punti_da_posizione(posizione) #associo i punti alla poszione del team
        punteggi_team[team] = punteggi_team.get(team, 0) + punti #sommo mano mano i p

    classifica = sorted(punteggi_team.items(), key=lambda x: x[1], reverse=True) #ordino i punteggi per creare una classifica
    for team, punti in classifica:
        print(f"{team}: {punti}") #stampo la classifica

classifica_team(dati)

Ferrari: 172
Mclaren: 151
Bmw: 135
Renault: 61
Toyota: 56
Toro Rosso: 35
